# Notebook 2: Frozen Embedding Baseline

**Run `python run_preprocessing.py` first** to generate the processed .pkl files.

Level 1: use ESM2 and ProtT5 as frozen feature extractors, then train XGBoost / LightGBM / MLP classifiers.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import pickle
import torch
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModel, T5EncoderModel
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
import xgboost as xgb
import lightgbm as lgb

from preprocessing import load_processed
from evaluate import compute_fmax, compute_aupr, load_ia_weights
from dataset import ESM2Dataset, ProtT5Dataset, collate_fn_pad

sns.set_style('whitegrid')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

IA_PATH = '../data/raw/IA.tsv'

## 1. Load Preprocessed Split

In [ ]:
ONTOLOGY = 'BPO'
data = load_processed(f'../data/processed/cafa6_{ONTOLOGY.lower()}.pkl')
go_terms = data['go_terms']
print(f'Ontology: {ONTOLOGY} | GO terms: {len(go_terms):,}')
print(f'Train: {len(data["train"]["protein_ids"]):,} | Val: {len(data["val"]["protein_ids"]):,} | Test: {len(data["test"]["protein_ids"]):,}')

# Load IA weights for Smin
ia_weights = load_ia_weights(IA_PATH, go_terms)
print(f'IA weights loaded: {(ia_weights > 0).sum()} / {len(go_terms)} terms have IA > 0')

## 2. Extract ESM2 Embeddings (Frozen)

In [ ]:
ESM2_MODEL = 'facebook/esm2_t6_8M_UR50D'  # 8M params — runs on CPU in minutes

tokenizer_esm2 = AutoTokenizer.from_pretrained(ESM2_MODEL)
backbone_esm2  = AutoModel.from_pretrained(ESM2_MODEL).to(DEVICE).eval()

@torch.no_grad()
def extract_esm2_embeddings(sequences, batch_size=32, max_length=512):
    dataset = ESM2Dataset(sequences, tokenizer_esm2, max_length)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_pad)
    embeddings = []
    for batch in tqdm(loader, desc='ESM2 embed'):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        out  = backbone_esm2(input_ids=ids, attention_mask=mask)
        m    = mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1e-9)
        embeddings.append(pooled.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

X_train = extract_esm2_embeddings(data['train']['sequences'])
X_val   = extract_esm2_embeddings(data['val']['sequences'])
X_test  = extract_esm2_embeddings(data['test']['sequences'])
print(f'Embedding shape: {X_train.shape}')

## 3. Train Classifiers

We cap at `N_TERMS` for speed; remove the cap to train on all GO terms (slow on CPU).

In [ ]:
N_TERMS = min(200, len(go_terms))   # increase for full training

Y_tr = data['train']['labels'][:, :N_TERMS]
Y_v  = data['val']['labels'][:, :N_TERMS]
Y_ts = data['test']['labels'][:, :N_TERMS]
ia_w = ia_weights[:N_TERMS]

results = {}

# --- Logistic Regression ---
print('Training Logistic Regression...')
lr = MultiOutputClassifier(LogisticRegression(max_iter=300, C=1.0, solver='saga'), n_jobs=-1)
lr.fit(X_train, Y_tr)
lr_scores = np.column_stack([e.predict_proba(X_val)[:, 1] for e in lr.estimators_])
fmax_lr, t_lr = compute_fmax(Y_v, lr_scores)
aupr_lr = compute_aupr(Y_v, lr_scores)
results['LogReg'] = {'Fmax': fmax_lr, 'AUPR': aupr_lr, 'Threshold': t_lr}
print(f'LogReg  | Fmax={fmax_lr:.4f} | AUPR={aupr_lr:.4f}')

In [ ]:
# --- LightGBM ---
print('Training LightGBM...')
lgb_clf = MultiOutputClassifier(
    lgb.LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, n_jobs=-1, verbose=-1),
    n_jobs=1
)
lgb_clf.fit(X_train, Y_tr)
lgb_scores = np.column_stack([e.predict_proba(X_val)[:, 1] for e in lgb_clf.estimators_])
fmax_lgb, t_lgb = compute_fmax(Y_v, lgb_scores)
aupr_lgb = compute_aupr(Y_v, lgb_scores)
results['LightGBM'] = {'Fmax': fmax_lgb, 'AUPR': aupr_lgb, 'Threshold': t_lgb}
print(f'LightGBM| Fmax={fmax_lgb:.4f} | AUPR={aupr_lgb:.4f}')

In [ ]:
# --- XGBoost ---
print('Training XGBoost...')
xgb_clf = MultiOutputClassifier(
    xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                      eval_metric='logloss', n_jobs=-1, verbosity=0),
    n_jobs=1
)
xgb_clf.fit(X_train, Y_tr)
xgb_scores = np.column_stack([e.predict_proba(X_val)[:, 1] for e in xgb_clf.estimators_])
fmax_xgb, t_xgb = compute_fmax(Y_v, xgb_scores)
aupr_xgb = compute_aupr(Y_v, xgb_scores)
results['XGBoost'] = {'Fmax': fmax_xgb, 'AUPR': aupr_xgb, 'Threshold': t_xgb}
print(f'XGBoost | Fmax={fmax_xgb:.4f} | AUPR={aupr_xgb:.4f}')

## 4. Results

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('Fmax', ascending=False)
print(results_df.to_string(float_format='{:.4f}'.format))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
results_df['Fmax'].plot.bar(ax=axes[0], color='steelblue', title=f'Fmax — {ONTOLOGY} (Val)')
results_df['AUPR'].plot.bar(ax=axes[1], color='coral',     title=f'AUPR — {ONTOLOGY} (Val)')
for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(f'../reports/baseline_results_{ONTOLOGY.lower()}.png', dpi=150)
plt.show()